# 11 · Wrap-up and take-homes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/11-wrap-up-and-take-homes.ipynb)

*wrap-up · 5 min*

> 🇪🇸 **Cierre y ejercicios para casa** — Qué conecta los bloques 4, 5 y 6, más tres ejercicios para casa.

What connects Blocks 4, 5 and 6, plus three take-home exercises.

## What you will be able to do

- State the one idea that connects the pseudoinverse, deconvolution and Tucker.
- Find the scaling trap in PCA on real, unstandardized data (take-home A).
- Build attention out of two contractions, and mask padded positions (take-home B).
- Run a real CP decomposition and read its components as trip types nobody labelled (take-home C).
- Trade parameter count against reconstruction error with a rank slider, on a real dense tensor (optional appendix).

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

rng = np.random.default_rng(0)

## What you did today

> 🇪🇸 Lo que hiciste hoy.

1. **Section 01** — learned the vocabulary of tensors (axis, order, shape, slice,
   fiber, unfolding, contraction, decomposition), and that unfolding turns any
   tensor into a matrix without losing anything.
2. **Sections 02 and 05** — argued about what axes *mean*, and found that a batch
   axis and a time axis behave differently even when the shapes look identical.
3. **Sections 03 and 04** — indexed, broadcast, reshaped and transposed real
   tumour data and real medical images, and hit real problems: zero-variance
   pixels, and reshape silently destroying an image.
4. **Sections 06–10** — wrote contractions with `einsum`; solved an unsolvable
   20,433-equation system with the pseudoinverse; used recursion to forecast real
   airline traffic and to find an eigenvector; convolved and deconvolved a real
   photograph; and compressed a real taxi tensor 4.7× with Tucker, which found
   rush hour on its own.

### One idea connects sections 07, 09 and 10

**When a problem has no exact answer or no true inverse, you do not give up —
you find the best stable approximation.** The pseudoinverse does this for linear
systems, Richardson-Lucy for blurred images, and Tucker for tensors that are too
large to keep in full.

> 🇪🇸 Cuando un problema no tiene respuesta exacta ni inversa verdadera, no te
> rindes: buscas la mejor aproximación estable.

## Where to go next

- `torch.einsum` / `tf.einsum` / `jnp.einsum` — **identical syntax** to what you
  used today.
- [`tensorly`](https://tensorly.org) — proper Tucker and CP decompositions.
- `np.linalg` — the rest of Chapter 2: eigendecomposition, `lstsq`, `pinv`, `qr`.
- `scipy.signal` and `skimage.restoration` — convolution and deconvolution
  beyond today.
- The three take-homes below.

### Optional: the same contraction in PyTorch

Everything today was NumPy, because that is what the workshop's real datasets
and verified numbers are built on. The einsum string does not change when you
move to a deep learning framework — only the array type does.

In [ ]:
# Optional. Colab has torch pre-installed; skip this cell if you prefer.
try:
    import torch
    photo = rng.standard_normal((8, 8, 3))
    w = np.array([0.2125, 0.7154, 0.0721])

    np_gray = np.einsum('hwc,c->hw', photo, w)
    pt_gray = torch.einsum('hwc,c->hw', torch.tensor(photo), torch.tensor(w))

    print(np.allclose(np_gray, pt_gray.numpy()))     # True — same string, same answer
except ImportError:
    print("torch not installed — nothing here you need")

---

## Take-home A — How many principal components are enough?

> 🇪🇸 Ejercicio para casa A: ¿cuántas componentes principales bastan?

**Real data contains a trap here. Find it.**

In [ ]:
bc = load_breast_cancer(); X, y = bc.data, bc.target

# TODO 1: Center X, run np.linalg.svd, and compute the fraction of variance each
#         component explains (variance is proportional to S**2).

# TODO 2: How many components explain 95% of the variance? The answer will look
#         TOO GOOD. Do not trust it yet.

# TODO 3: Print X.var(axis=0). The 30 measurements use different units — some are
#         areas in the thousands, some are ratios below 1. What is that doing?

# TODO 4: Redo everything on standardized data: (X - mean) / std. How many now?

# TODO 5: Scatter-plot the first 2 components, coloured by y. Do the two groups
#         separate?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
Xc = X - X.mean(axis=0)
S = np.linalg.svd(Xc, full_matrices=False)[1]
frac = S**2 / (S**2).sum()
n95 = np.argmax(np.cumsum(frac) >= 0.95) + 1      # 1  (!)
print(n95, round(frac[0], 3))                      # 1 0.982

print(np.sort(X.var(axis=0))[[0, -1]])             # ~0.0000075 up to ~324000

Xs = (X - X.mean(axis=0)) / X.std(axis=0)
S2 = np.linalg.svd(Xs, full_matrices=False)[1]
n95_scaled = np.argmax(np.cumsum(S2**2 / (S2**2).sum()) >= 0.95) + 1   # 10
print(n95_scaled)

# Without standardizing, the first component appears to explain 98.2% of the
# variance. IT IS AN ILLUSION: `worst area` has a variance around 323,000 while
# smoothness values sit below 1, so PCA reports the largest UNIT, not the
# largest PATTERN. After standardizing, the first component explains 44% and
# TEN components are needed.
#
# PCA KNOWS NOTHING ABOUT UNITS. Features on different scales must be
# standardized first.

# TODO 5:
# Z = Xs @ np.linalg.svd(Xs, full_matrices=False)[2][:2].T
# import matplotlib.pyplot as plt
# plt.scatter(Z[:, 0], Z[:, 1], c=y, s=8, cmap="coolwarm")

---

## Take-home B — Attention is two contractions

> 🇪🇸 Ejercicio para casa B: la atención son dos contracciones.

Attention is the mechanism that answers question 5 from section 05: *which parts
of a sequence matter most?* Protein language models use it so every amino acid
can look at every other one; recommenders use it to weight a user's past
interactions.

In [ ]:
np.random.seed(6)
batch, seq_len, dim = 4, 12, 16
Q, K, V = (np.random.randn(batch, seq_len, dim) for _ in range(3))

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

# TODO 1: With einsum, compute scores[b,i,j] = how much position i attends to
#         position j. Shape (4, 12, 12). Scale by 1/sqrt(dim).

# TODO 2: Apply softmax on the correct axis so each row of weights sums to 1.

# TODO 3: With einsum, combine V using those weights -> (4, 12, 16).

# TODO 4: Suppose the last 3 positions are padding, not real data. Build a mask,
#         set those scores to -np.inf BEFORE the softmax, and verify the padded
#         positions receive exactly zero weight.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
scores  = np.einsum('bid,bjd->bij', Q, K) / np.sqrt(dim)
weights = softmax(scores, axis=-1)
output  = np.einsum('bij,bjd->bid', weights, V)
print(scores.shape, weights.shape, output.shape)
print(np.allclose(weights.sum(axis=-1), 1.0))        # True

mask = np.zeros((seq_len, seq_len)); mask[:, -3:] = -np.inf
weights_masked = softmax(scores + mask, axis=-1)
print(weights_masked[..., -3:].max())                # 0.0 — exactly zero weight

# `scores` is Chapter 2's dot product (eq. 2.8); `output` is Chapter 2's linear
# combination (eq. 2.28). ATTENTION IS TWO CONTRACTIONS built from ideas you had
# already read.
#
# TODO 4 solves the variable-length problem from section 02: THE MASK IS HOW
# REAL MODELS HANDLE SEQUENCES AND VIDEOS OF DIFFERENT LENGTHS.

---

## Take-home C — CP decomposition, compared to Tucker

> 🇪🇸 Ejercicio para casa C: CP comparado con Tucker.

In [ ]:
# TODO 1: Build one rank-1 tensor with einsum from three random vectors of
#         length 4, 5 and 24. What shape is it? How many numbers define it?

# TODO 2: Compare that against 4*5*24. What is the compression of ONE rank-1 piece?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
a, b, c = rng.standard_normal(4), rng.standard_normal(5), rng.standard_normal(24)
rank1 = np.einsum('i,j,k->ijk', a, b, c)     # (4, 5, 24) from only 33 numbers
print(rank1.shape, len(a) + len(b) + len(c), 4 * 5 * 24)   # (4,5,24) 33 480
print(round(480 / 33, 1))                                   # 14.5x for one piece

# A full CP decomposition is a SUM of R pieces like this one, not just a single
# rank-1 term. The cells below build a real rank-3 CP model on real data — no
# more commented-out pseudocode.

## Now decompose a real tensor with CP

> 🇪🇸 Ahora sí: una descomposición CP real sobre un tensor real.

This take-home is separate from section 10's notebook, so it rebuilds the same
real taxi tensor here rather than assuming section 10 already ran.

In [ ]:
TAXIS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv"
taxis = pd.read_csv(TAXIS)
taxis['hour'] = pd.to_datetime(taxis['pickup']).dt.hour
sub = taxis.dropna(subset=['pickup_borough', 'dropoff_borough'])
pb = sorted(sub['pickup_borough'].unique())
db = sorted(sub['dropoff_borough'].unique())

T = np.zeros((len(pb), len(db), 24))
for (p, d, h), v in sub.groupby(['pickup_borough', 'dropoff_borough', 'hour']).size().items():
    T[pb.index(p), db.index(d), h] = v

print(T.shape, pb, db)   # (4, 5, 24) — the same real taxi tensor as section 10,
                         # rebuilt here so this notebook stands on its own

CP needs a library here rather than the by-hand HOSVD from section 10: an ALS
loop short enough to read is also too short to be a reliable optimizer, and
getting that wrong would teach the wrong lesson. [`tensorly`](https://tensorly.org)
is not part of Colab's default image, so the install is explicit, the same way
section 10 tells you it borrowed the idea from a real library rather than
hiding it.

> 🇪🇸 CP necesita aquí una librería en vez del HOSVD hecho a mano de la sección
> 10: un bucle ALS lo bastante corto para leerse también es demasiado corto
> para ser un optimizador confiable. `tensorly` no viene instalado por defecto
> en Colab, así que la instalación es explícita.

In [ ]:
%pip install -q tensorly

import tensorly as tl
from tensorly.decomposition import parafac

R = 3   # three real, checkable trip patterns fit this tensor's size
cp_weights, cp_factors = parafac(tl.tensor(T), rank=R, init='svd',
                                  random_state=0, n_iter_max=500, tol=1e-9)
Fpb, Fdb, Fhr = cp_factors                       # (4, 3), (5, 3), (24, 3)

cp_recon = tl.cp_to_tensor((cp_weights, cp_factors))
cp_error = np.linalg.norm(cp_recon - T) / np.linalg.norm(T)
print(f"CP rank {R}: relative reconstruction error = {cp_error:.3f}")
print("Section 10's Tucker, rank (2, 2, 3), measured 0.067 on this same tensor.")

## What CP's uniqueness buys you, and what it does not

> 🇪🇸 Lo que la unicidad de CP te da, y lo que no te da.

PCA and Tucker's factor matrices are only defined up to an arbitrary rotation
within each subspace of similar size — ask for the "second principal
component" of near-equal-variance data and the answer is unstable. **CP has no
such freedom**, under a condition on the factor matrices called the Kruskal
condition, which this tensor satisfies. A CP component is only free to move in
three limited ways: the three components can be listed in any **order**; a
scalar can move between the three factor vectors of one component as long as
their **product** is unchanged; and because these are real (not just
positive) numbers, an even number of those factors can flip **sign** together.
None of that changes what one component *looks like* — it is still one
coherent pattern per axis, not a rotated mixture of several. That is why the
components below are worth reading individually, and why the code below uses
`abs()` before asking which entry is strongest — the strongest entry does not
move, only its sign might.

**Analysts benefit because CP exposes one interpretable pattern per axis —
pickup, dropoff and hour together — that can be read as a coherent trip type,
the way PCA's freely-rotating components cannot be.**

In [ ]:
# Colab renders ipywidgets through its own widget manager rather than the
# classic Jupyter one; this call is a no-op outside Colab, which is why it is
# guarded rather than assumed.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

import ipywidgets as widgets
import matplotlib.pyplot as plt

def show_component(component):
    r = component - 1   # the slider shows 1..R for students; factors are 0-indexed
    plt.close('all')
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
    axes[0].bar(pb, Fpb[:, r], color='#4C72B0')
    axes[0].set_title('Pickup borough'); axes[0].tick_params(axis='x', rotation=40)
    axes[1].bar(db, Fdb[:, r], color='#DD8452')
    axes[1].set_title('Dropoff borough'); axes[1].tick_params(axis='x', rotation=40)
    axes[2].bar(range(24), Fhr[:, r], color='#55A868')
    axes[2].set_title('Hour of day'); axes[2].set_xlabel('hour')
    fig.suptitle(f'CP component {component} of {R}')
    plt.tight_layout()
    plt.show()

    print(f"Strongest pickup borough:  {pb[np.argmax(np.abs(Fpb[:, r]))]}")
    print(f"Strongest dropoff borough: {db[np.argmax(np.abs(Fdb[:, r]))]}")
    print(f"Peak hour:                 {int(np.argmax(np.abs(Fhr[:, r])))}")

# TODO 3: Flip through all three components (1, 2, 3). Does each one read as
#         a different, nameable kind of trip? Which hour is each one busiest?
widgets.interact(show_component,
                  component=widgets.IntSlider(min=1, max=R, step=1, value=1,
                                               description='Component'));

---

## After the workshop — Tucker compression for deployment

> 🇪🇸 Después del taller — compresión de Tucker para producción.

**Optional — run this after the workshop.** Section 10 ran one fixed Tucker
rank. Here a **rank slider** drives the trade-off live, on a real dense array,
so you can feel the curve instead of reading one number on it.

One honest note before the code: this is **not** a neural network's weights.
A small, stable, seconds-to-download real conv-weight file that both fits a
free Colab CPU and is not already engineered to be maximally compact turned
out not to exist — the two real options checked while building this notebook
(a modern efficient architecture, and a small classifier trained from scratch
on this workshop's own data) were **already so parameter-efficient that Tucker
found almost nothing left to compress**, which is itself real and worth
knowing, just not the point of this appendix. So instead this is a
**comparable dense tensor**: real NYC taxi trips again, but counted over
**pickup borough × dropoff borough × hour × weekday** — a genuine order-4
array, the same shape of thing an on-device cache or a recommender's usage
table has to fit in memory. The Tucker math, the slider, and the trade-off it
shows are identical to compressing a weight tensor; only the source of the
numbers differs, and it seemed better to say that plainly than to relabel taxi
trips as something they are not.

**Deployment engineers benefit because Tucker lets them choose a point on this
curve explicitly** — cut most of an array's storage and pay only a measured,
bounded increase in error, rather than guessing at a fixed compression
level.

In [ ]:
TAXIS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv"
taxis = pd.read_csv(TAXIS)
taxis['hour'] = pd.to_datetime(taxis['pickup']).dt.hour
taxis['weekday'] = pd.to_datetime(taxis['pickup']).dt.weekday
sub = taxis.dropna(subset=['pickup_borough', 'dropoff_borough'])
pb2 = sorted(sub['pickup_borough'].unique())
db2 = sorted(sub['dropoff_borough'].unique())

demand = np.zeros((len(pb2), len(db2), 24, 7))
for (p, d, h, wd), v in sub.groupby(
        ['pickup_borough', 'dropoff_borough', 'hour', 'weekday']).size().items():
    demand[pb2.index(p), db2.index(d), h, wd] = v

print(demand.shape, int(demand.sum()))   # (4, 5, 24, 7), same trips as above

In [ ]:
def unfold(T, axis):
    return np.moveaxis(T, axis, 0).reshape(T.shape[axis], -1)

# Precompute BOTH SVD bases once. The slider below only re-slices and
# re-contracts these small matrices — it never redoes an SVD, which is what
# keeps it responsive. Only the two time axes are compressed; pickup and
# dropoff borough stay exact, the way section 10's kernel spatial dims would
# stay exact in a channel-mode Tucker compression of a real conv layer.
basis_hour    = np.linalg.svd(unfold(demand, 2), full_matrices=False)[0]   # (24, 24)
basis_weekday = np.linalg.svd(unfold(demand, 3), full_matrices=False)[0]   # (7, 7)

n_pb, n_db, n_hour, n_weekday = demand.shape
original_params = demand.size

# Sweep every achievable rank once, up front, so the widget only ever looks
# values up rather than recomputing them.
ranks = list(range(1, n_hour + 1))
compressed_list, ratio_list, error_list, madds_list = [], [], [], []
for k in ranks:
    r_hour, r_weekday = k, min(k, n_weekday)
    Uh, Uw = basis_hour[:, :r_hour], basis_weekday[:, :r_weekday]
    core  = np.einsum('ijhw,hc,wd->ijcd', demand, Uh, Uw)
    recon = np.einsum('ijcd,hc,wd->ijhw', core, Uh, Uw)
    compressed = core.size + Uh.size + Uw.size
    compressed_list.append(compressed)
    ratio_list.append(original_params / compressed)
    error_list.append(np.linalg.norm(demand - recon) / np.linalg.norm(demand))
    # Multiply-adds to RE-EXPAND the compressed factors back to the full
    # array — the cost a deployed system pays each time it reads the cache.
    # This is not a network FLOP count; it is specifically that one contraction.
    madds_list.append(n_pb * n_db * n_hour * r_weekday * (r_hour + n_weekday))

print(f"original parameters: {original_params} "
      f"(pickup {n_pb} x dropoff {n_db} x hour {n_hour} x weekday {n_weekday})")

In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

import ipywidgets as widgets
import matplotlib.pyplot as plt

def tucker_tradeoff(k):
    i = k - 1
    plt.close('all')
    fig, ax1 = plt.subplots(figsize=(6.5, 3.2))
    ax1.plot(ranks, error_list, color='#C44E52')
    ax1.scatter([k], [error_list[i]], color='#C44E52', zorder=5)
    ax1.set_xlabel('rank k (shared by the hour and weekday axes)')
    ax1.set_ylabel('relative error', color='#C44E52')
    ax2 = ax1.twinx()
    ax2.plot(ranks, ratio_list, color='#4C72B0')
    ax2.scatter([k], [ratio_list[i]], color='#4C72B0', zorder=5)
    ax2.set_ylabel('compression ratio (x)', color='#4C72B0')
    plt.tight_layout()
    plt.show()

    print(f"rank k = {k}")
    print(f"compressed parameters: {compressed_list[i]}  (of {original_params} original)")
    print(f"compression ratio:     {ratio_list[i]:.2f}x")
    print(f"relative error:        {error_list[i]:.3f}")
    print(f"reconstruction MAdds:  {madds_list[i]}  "
          f"(multiply-adds to re-expand the factors back to the full array)")

# Move the slider from 1 to n_hour. Both ends are worth visiting: rank 1 is
# the cheapest possible model, and the top end (hour AND weekday both at
# their true dimension) should reconstruct the tensor exactly — a check on
# the implementation, not just on the trade-off.
widgets.interact(tucker_tradeoff,
                  k=widgets.IntSlider(min=1, max=n_hour, step=1, value=4,
                                       description='rank k'));

## Thank you

> 🇪🇸 Gracias por venir. Pregunta en Discord en español o en inglés — lo que te
> permita preguntar más rápido.

Questions stay welcome in Discord, in Spanish or English. The
[handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)
has everything from today, including the facilitator notes.

---

## Done with this section

That is the whole workshop. Thank you for coming.

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)